# V70 RESUBMIT — Metric Fix Validation

**Objetivo**: testar se o V70 adapter atual (que deu 0.84 local) pode render **0.86-0.89** no Kaggle com a metric fix aplicada.

**Background**: Agent D1 descobriu que nosso `scripts/local_score.py` tinha 3 BUGS:
1. Strict bit regex `re.fullmatch(r'[01]+')` — metric oficial NÃO faz
2. `abs(a-b)<1e-2` em vez de `math.isclose(rel_tol=1e-2)`  — 1000x mais estrito
3. Faltava `enable_thinking=True` no chat template

**Agent D1 PART 2 simulação em 9500 rows**:
- V70 local old metric: 0.84
- V70 Kaggle real estimado (conservador): **0.86** (+2pp)
- V70 Kaggle real estimado (liberal): **0.89** (+5pp)

**Custo**: 1 Kaggle submit slot (5/dia)

**Riscos**:
- Adapter variance natural: ±0.01-0.02 entre runs
- Se ≥0.86: **confirma metric fix ajuda** + próximo passo V71.x
- Se ≥0.87: **JÁ SOMOS TOP 1**
- Se <0.84: variance regredir (2/5 submit slots reservados)

In [ ]:
# Cell 1 — Mount GDrive (adapter V70 está aqui)
from google.colab import drive, userdata
drive.mount('/content/drive')

# Localizar V70 adapter no GDrive
import subprocess
print('Procurando V70 adapter no GDrive...')
result = subprocess.run(
    ['find', '/content/drive/MyDrive', '-iname', 'adapter_model.safetensors'],
    capture_output=True, text=True, timeout=60
)
adapters = [l for l in result.stdout.strip().split('\n') if l]
print(f'Encontrados {len(adapters)} adapters:')
for a in adapters:
    print(f'  {a}')

# Identificar V70 por data/nome
V70_PATH = None
for a in adapters:
    if 'v70' in a.lower() or 'huikang' in a.lower() or 'C4EA449A' in a:
        V70_PATH = str(a).rsplit('/', 1)[0]
        print(f'\nV70 candidato: {V70_PATH}')
        break

if V70_PATH is None and adapters:
    # Manual fallback
    print('\n⚠️  Multiple adapters found. Set V70_PATH manually:')
    V70_PATH = adapters[0].rsplit('/', 1)[0]
    print(f'Default: {V70_PATH}')

assert V70_PATH, 'NO adapter found in GDrive — upload V70 first'
print(f'\n✓ Using V70_PATH = {V70_PATH}')

In [ ]:
# Cell 2 — Verify adapter integrity + SHA check
import json, hashlib
from pathlib import Path

adapter_path = Path(V70_PATH)
cfg_file = adapter_path / 'adapter_config.json'
weights_file = adapter_path / 'adapter_model.safetensors'

assert cfg_file.exists(), f'adapter_config.json not found in {adapter_path}'
assert weights_file.exists(), f'adapter_model.safetensors not found in {adapter_path}'

# Read config
cfg = json.loads(cfg_file.read_text())
print('Adapter config:')
print(f'  r: {cfg.get("r")}')
print(f'  lora_alpha: {cfg.get("lora_alpha")}')
print(f'  target_modules: {cfg.get("target_modules")}')
print(f'  base_model: {cfg.get("base_model_name_or_path")}')

# SHA256 check (first 8 chars)
h = hashlib.sha256(weights_file.read_bytes()).hexdigest()
print(f'\nadapter_model.safetensors SHA256[:8]: {h[:8]}')
print(f'Size: {weights_file.stat().st_size / 1024**2:.1f} MB')

# Check if this is V70 (SHA prefix C4EA449A per plan)
if h[:8].lower() == 'c4ea449a':
    print('\n✅ CONFIRMED: This is V70 (SHA C4EA449A)')
else:
    print(f'\n⚠️  SHA differs from known V70 (C4EA449A). This may be a different training.')
    print(f'    Proceed anyway? User confirmation needed.')

In [ ]:
# Cell 3 — Clone KG1 repo with metric-fixed scripts
import subprocess
repo_url = 'https://github.com/felipesp1983-work/KG1-NVIDIA.git'  # OR user's fork
repo_path = '/content/kg1'

# User should set their token if private
# github_token = userdata.get('GITHUB_TOKEN')
# auth_url = repo_url.replace('https://', f'https://{github_token}@')

# Alternative: rsync from the worktree where metric fix was applied
# User can zip the worktree, upload to GDrive, unzip here

# For now, copy scripts directly via paste
print('⚠️ MANUAL STEP: Copy these files from worktree to /content/kg1/scripts/:')
print('  - scripts/local_score.py (metric FIXED 2026-04-21)')
print('  - scripts/kg1_local_metric_gate.py (metric FIXED)')
print('  - scripts/submit_kaggle.py')
print('  - scripts/kg1_submission_gate.py')
print('\nOR zip worktree locally, upload to GDrive, extract here:')
print('  !unzip /content/drive/MyDrive/kg1_worktree.zip -d /content/kg1/')

# Assume done:
import sys
sys.path.insert(0, '/content/kg1')

In [ ]:
# Cell 4 — Local gate with CORRECTED metric (D1 fix)
# This tests if V70 passes our CORRECTED local eval
# Expected: higher score than old buggy metric

import subprocess, pandas as pd

result = subprocess.run([
    'python', '/content/kg1/scripts/local_score.py',
    '--adapter', V70_PATH,
    '--n-samples', '600',
    '--output-csv', '/content/v70_local_eval_metric_fixed.csv',
], capture_output=True, text=True, timeout=3600)

print('STDOUT (last 3000 chars):')
print(result.stdout[-3000:])
print('\nSTDERR (last 1000 chars):')
print(result.stderr[-1000:])

# Load results
try:
    df = pd.read_csv('/content/v70_local_eval_metric_fixed.csv')
    overall = df['correct'].mean()
    print(f'\n🎯 V70 local eval with METRIC FIXED: {overall:.4f}')
    print('(Compare to old buggy local: 0.84)')
    print()
    if 'category' in df.columns:
        print('Per category:')
        print(df.groupby('category')['correct'].mean().sort_values(ascending=False))
except Exception as e:
    print(f'Eval failed: {e}')

In [ ]:
# Cell 5 — Prepare submission ZIP (V70 unchanged, just repackage)
import shutil, zipfile
from pathlib import Path

submission_dir = Path('/content/v70_submission')
submission_dir.mkdir(exist_ok=True)

# Copy adapter files
shutil.copy(V70_PATH + '/adapter_config.json', submission_dir / 'adapter_config.json')
shutil.copy(V70_PATH + '/adapter_model.safetensors', submission_dir / 'adapter_model.safetensors')

# Validate
cfg = json.loads((submission_dir / 'adapter_config.json').read_text())
assert cfg.get('r') == 32, f'r must be 32, got {cfg.get("r")}'
assert 'in_proj' in cfg.get('target_modules', []), 'in_proj required'

# Zip
zip_path = Path('/content/v70_resubmit.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in submission_dir.glob('*'):
        zf.write(f, arcname=f.name)

print(f'ZIP created: {zip_path}')
print(f'Size: {zip_path.stat().st_size / 1024**2:.1f} MB')
print(f'Contents: {[n.filename for n in zipfile.ZipFile(zip_path).infolist()]}')

In [ ]:
# Cell 6 — Kaggle submission gate (sanity check)
import subprocess

result = subprocess.run([
    'python', '/content/kg1/scripts/kg1_submission_gate.py',
    '--adapter-zip', '/content/v70_resubmit.zip',
    '--test-csv', '/content/kg1/data/kaggle/unzipped/test.csv',  # or Kaggle input path
    '--fail-on-block',
], capture_output=True, text=True)

print(result.stdout[-2000:])
print('---STDERR---')
print(result.stderr[-500:])

assert result.returncode == 0, 'Gate BLOCKED — do not submit'
print('\n✅ Gate PASS — safe to submit')

In [ ]:
# Cell 7 — Submit to Kaggle (CONSUMES 1/5 daily slot)
import subprocess, os

# Kaggle creds
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME') or 'felipe1983'
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
assert os.environ['KAGGLE_KEY'], 'KAGGLE_KEY required in Colab secrets'

# Final safety check — user confirmation
CONFIRM = input('Confirm submit V70 to Kaggle? (yes/no): ').strip().lower()
assert CONFIRM == 'yes', 'Aborted by user'

result = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'nvidia-nemotron-model-reasoning-challenge',
    '-f', '/content/v70_resubmit.zip',
    '-m', 'V70 resubmit — metric correction D1 test (2026-04-21)',
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

# Wait ~3 min for scoring
print('\n⏳ Kaggle scoring takes ~5-15 min. Check LB:')
print('  kaggle competitions submissions -c nvidia-nemotron-model-reasoning-challenge')

In [ ]:
# Cell 8 — Monitor submission status
import subprocess, time

for attempt in range(20):
    time.sleep(60)  # wait 60s
    result = subprocess.run([
        'kaggle', 'competitions', 'submissions',
        '-c', 'nvidia-nemotron-model-reasoning-challenge',
        '--csv',
    ], capture_output=True, text=True)
    
    # Parse CSV
    import csv, io
    reader = csv.DictReader(io.StringIO(result.stdout))
    latest = next(reader, None)
    if latest:
        status = latest.get('status', '?')
        score = latest.get('publicScore', '?')
        print(f'[{attempt+1}/20] status={status} score={score}')
        if status.lower() in ['complete', 'failed', 'error']:
            print(f'\n🎯 FINAL STATUS: {status}')
            print(f'🎯 SCORE: {score}')
            if score != '?' and float(score) >= 0.87:
                print('🏆 TOP 1 CONFIRMED!')
            elif score != '?' and float(score) >= 0.86:
                print('✅ Plateau atingido! Continue V71+ roadmap')
            elif score != '?' and float(score) >= 0.84:
                print('⚖️ No regresssion. Metric fix impact menor que simulação.')
            else:
                print('⚠️ REGRESSION — review submission or variance')
            break
else:
    print('⏱️ Timeout — check manually')

## 🎯 Next Steps Based on Outcome

### If score ≥ 0.87 (TOP 1 achieved!)
1. **Celebrate** 🎉
2. Monitor LB for 48h — confirm ranking holds
3. Prepare writeup required for prize eligibility
4. Optional: continue V71+ for margin

### If score ≥ 0.86 (plateau reached)
1. Gap 0.86→0.87 = 0.01pp
2. V71.2 (bit pairs) + V71.3 (cryptarithm) = **+0.010-0.020** → should break plateau
3. Run `scripts/v71_prepare_training_data.py --include-programmatic` locally
4. Train V71 notebook — see `KG1_V70_5_FIXED_METRIC.ipynb`

### If score 0.84-0.86 (moderate impact)
- Metric fix confirmed working but model needs improvements
- Proceed with V71+ full roadmap (V70.5 → V71.x → V72 → V72.5)

### If score < 0.84 (regression)
- Natural variance (±0.02 known)
- Retry once (1 more slot)
- Se persistir: investigar submission integrity (zip correto, adapter config, etc)
